In [ ]:
%rm -rf bench_research_ml_project
!git clone https://github.com/oremaz/bench_research_ml_project
%cd bench_research_ml_project
!pip install -r requirements.txt
!pip uninstall numpy scipy scikit-learn -y
!pip install numpy==1.26.3 scipy==1.16.2 scikit-learn==1.7.2
!pip install transformers==4.51.3
!pip install bitsandbytes==0.47.0 trl
!pip install -q timm==1.0.9 kaggle rich


# Recipe Classification/Regression Benchmark Report

This notebook benchmarks data augmentation methods and model architectures for recipe classification/regression, using robust pipelines and external evaluation. For each task and model, training, saving, evaluation, and visualization are performed and results are saved for reproducibility.


In [ ]:
%cd ml_pipeline

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
import torch
from utils.data import load_csv, prepare_embeddings_data, train_val_test_split, LabelEncoderHelper, filter_meal_types
from utils.metrics import METRIC_REGISTRY
from utils.visualization import plot_confusion_matrix, plot_regression_results
from pipelines_torch.models import MODEL_REGISTRY
from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.base import SimplePredictor
from data_augmentation.augmentations import AUGMENTATION_REGISTRY
from utils.visualization import plot_metrics_bar
from utils.utils import load_model, RESULTS_DIR

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## Difficulty Classification
### 1. Data Preprocessing
Load and preprocess data for difficulty classification.


In [ ]:
df = load_csv('recipes_df.csv')
df = df.dropna(subset=['embeddings_class', 'difficult'])
# Map 'A challenge' to 'More effort'
df['difficult'] = df['difficult'].replace({'A challenge': 'More effort'})
print('Unique difficulty labels after mapping:', df['difficult'].unique())
X, y = prepare_embeddings_data(df, target_column='difficult', embedding_column='embeddings_class')
le = LabelEncoderHelper(); le.fit(y)
y_enc = le.transform(y)
if callable(getattr(le, 'classes', None)):
    classes_list = le.classes()
else:
    classes_list = le.classes
label_map = dict(zip(classes_list, le.transform(classes_list)))
print('Label mapping:', label_map)
X_train, X_val, _, y_train, y_val, _ = train_val_test_split(X, y_enc, val_size=0.2, test_size=0, stratify=y_enc)
input_dim = X_train.shape[1]
print(input_dim)
num_classes = len(np.unique(y_train))
model_names = [
    'mlp_classifier', 'deep_mlp_classifier', 'random_forest_classifier', 'xgboost_classifier', 'lightgbm_classifier'
]
model_configs = [
    {
        'name': name,
        'class': MODEL_REGISTRY[name],
        'params': {'input_dim': input_dim, 'num_classes': num_classes} if 'mlp' in name or 'transformer' in name else {}
    }
    for name in model_names
]

# Difficulty task: use mixup_smote instead of smote, remove random_oversampler
augmentation_names = [
    'none',  'borderline_smote', 'svm_smote', 'kmeans_smote'
]
augmentations = [AUGMENTATION_REGISTRY[name] for name in augmentation_names]
unique, counts = np.unique(y_val, return_counts=True)
class_dist = dict(zip(unique, counts))
print("Validation set class distribution:")
for cls, cnt in class_dist.items():
    print(f"Class {cls}: {cnt}")

### 2. Augmentation Benchmarking
Benchmark augmentations using the first model. Select the best augmentation.


#### Mlp_model

In [ ]:
import os

# Define metrics for augmentation benchmarking
metrics = [METRIC_REGISTRY['f1'], METRIC_REGISTRY['recall'], METRIC_REGISTRY['precision'], METRIC_REGISTRY['accuracy']]

print("Running augmentation benchmark with class weights...")
print(f"Testing {len(augmentations)} augmentation methods on MLP classifier")

runner = BenchmarkRunner(
    model_configs=[model_configs[0]],
    augmentations=augmentations,
    metrics=metrics,
    task_type='classification',
    device='cpu',
    epochs=150,
    batch_size=32,
    early_stopping=20,
    use_class_weights=True, 
    dropout=0.3,
    weight_decay=5e-4, 
    learning_rate=1e-4,
    path_start='difficulty', 
    max_factor=1.0  
)
# Run training - results are automatically saved to results/difficulty/
runner.run(X, y_enc)

# Add comparison with use_class_weights=False for none augmentation
print("\nRunning none augmentation with use_class_weights=False for comparison...")
runner_no_weights = BenchmarkRunner(
    model_configs=[model_configs[0]],
    augmentations=[AUGMENTATION_REGISTRY['none']],  # Only none augmentation
    metrics=metrics,
    task_type='classification',
    device='cpu',
    epochs=150,
    batch_size=32,
    early_stopping=20,
    use_class_weights=False,  # This is the key difference
    dropout=0.3,
    weight_decay=5e-4, 
    learning_rate=1e-4,
    path_start='difficulty_no_weights'
)
# Run training - results are automatically saved
runner_no_weights.run(X, y_enc)

print("\n✅ Training complete! Results saved to:")
print("  - results/difficulty/")
print("  - results/difficulty_no_weights/")
print("\nYou can load and compare the metrics CSV files from these directories.")

In [ ]:
import os

# Use metrics already defined in previous cell
print("Running augmentation benchmark with max_factor=1.5...")
print(f"Testing {len(augmentations)} augmentation methods on MLP classifier")

runner = BenchmarkRunner(
    model_configs=[model_configs[0]],
    augmentations=augmentations,
    metrics=metrics,
    task_type='classification',
    device='cpu',
    epochs=150,
    batch_size=32,
    early_stopping=20,
    use_class_weights=False, 
    dropout=0.3,
    weight_decay=5e-4, 
    learning_rate=1e-4,
    path_start='difficulty2', 
    max_factor=1.5  
)
# Run training - results are automatically saved to results/difficulty2/
runner.run(X, y_enc)

print("\n✅ Training complete! Results saved to results/difficulty2/")
print("You can load and compare the metrics CSV files from this directory.")

#### Xgboost_model

In [ ]:
import os

# Use metrics already defined in previous cells
print("Running augmentation benchmark for XGBoost with class weights...")
print(f"Testing {len(augmentations)} augmentation methods on XGBoost classifier")

runner = BenchmarkRunner(
    model_configs=[model_configs[3]],
    augmentations=augmentations,
    metrics=metrics,
    task_type='classification',
    device='cpu',
    epochs=150,
    batch_size=32,
    early_stopping=20,
    use_class_weights=True, 
    dropout=0.3,
    weight_decay=5e-4, 
    learning_rate=1e-4,
    path_start='difficulty_xgb', 
    max_factor=1.0  
)
# Run training - results are automatically saved to results/difficulty_xgb/
runner.run(X, y_enc)

# Add comparison with use_class_weights=False for none augmentation
print("\nRunning none augmentation with use_class_weights=False for comparison...")
runner_no_weights = BenchmarkRunner(
    model_configs=[model_configs[3]],
    augmentations=[AUGMENTATION_REGISTRY['none']],  # Only none augmentation
    metrics=metrics,
    task_type='classification',
    device='cpu',
    epochs=150,
    batch_size=32,
    early_stopping=20,
    use_class_weights=False,  # This is the key difference
    dropout=0.3,
    weight_decay=5e-4, 
    learning_rate=1e-4,
    path_start='difficulty_xgb_no_weights'
)
# Run training - results are automatically saved
runner_no_weights.run(X, y_enc)

print("\n✅ Training complete! Results saved to:")
print("  - results/difficulty_xgb/")
print("  - results/difficulty_xgb_no_weights/")
print("\nYou can load and compare the metrics CSV files from these directories.")

In [ ]:
import os

# Use metrics already defined in previous cells
print("Running augmentation benchmark for XGBoost with max_factor=1.5...")
print(f"Testing {len(augmentations)} augmentation methods on XGBoost classifier")

runner = BenchmarkRunner(
    model_configs=[model_configs[3]],
    augmentations=augmentations,
    metrics=metrics,
    task_type='classification',
    device='cpu',
    epochs=150,
    batch_size=32,
    early_stopping=20,
    use_class_weights=False, 
    dropout=0.3,
    weight_decay=5e-4, 
    learning_rate=1e-4,
    path_start='difficulty2_xgb', 
    max_factor=1.5
)
# Run training - results are automatically saved to results/difficulty2_xgb/
runner.run(X, y_enc)

print("\n✅ Training complete! Results saved to results/difficulty2_xgb/")
print("You can load and compare the metrics CSV files from this directory.")


### 3. Training & Testing
Train and test all models using the best augmentation.


In [ ]:
print("Training models with BenchmarkRunner...")
print(f"Total dataset size: {len(X)} samples")
print(f"Number of models: {len(model_configs)}")
print()

# BenchmarkRunner will use the complete dataset X, y_enc
# and internally split for validation if use_kfold=False
runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],  # Start with no augmentation
    task_type="classification",
    device=DEVICE,
    epochs=120,
    batch_size=32,
    use_kfold=False,  # Set to False to use a single train/val split
    learning_rate=3e-4,
    path_start="difficulty_train",
    random_state=SEED,
)

# Run training - BenchmarkRunner handles splits internally
# It will save models and metrics automatically to results/difficulty_train/
runner.run(X, y_enc)

print("\nTraining complete! Models and metrics saved to results/difficulty_train/")

## QLoRA Fine-Tuning with Qwen3-4B-Instruct

This section demonstrates text-based difficulty classification using QLoRA (Quantized Low-Rank Adaptation) fine-tuning with the **Qwen3-4B-Instruct-2507** model.

**Key Differences from Embedding-Based Approach:**
- Uses raw recipe text (`recipe_text`) instead of pre-computed embeddings
- Leverages a 4-bit quantized language model for parameter-efficient fine-tuning
- Applies LoRA adapters to specific layers for task-specific learning
- Training is faster with QLoRA compared to full fine-tuning (10 epochs should be sufficient)

**Model Configuration:**
- Model: `Qwen/Qwen3-4B-Instruct-2507` (4B parameters, instruction-tuned)
- Quantization: 4-bit with BitsAndBytes for GPU efficiency
- LoRA: Low-rank adaptation for efficient fine-tuning
- Task: Multi-class classification (difficulty levels)

In [ ]:
# Prepare data for QLoRA text-based classification
# Note: Using recipe_text column, not embeddings
df_text = df[['recipe_text', 'difficult']].dropna()

# Check the distribution of difficulty labels
print(f"Dataset size: {len(df_text)}")
print(f"Difficulty distribution:\n{df_text['difficult'].value_counts()}")

# Get number of unique difficulty classes
num_classes = df_text['difficult'].nunique()
print(f"\nNumber of difficulty classes: {num_classes}")

# Prepare X and y for BenchmarkRunner
X_text = df_text['recipe_text'].values
y_difficult = df_text['difficult'].values

# Configure QLoRA model for difficulty classification
qwen_config = {
    'name': 'hf_qlora_classifier',
    'class': MODEL_REGISTRY['hf_qlora_classifier'],
    'params': {
        'model_name': 'Qwen/Qwen3-4B-Instruct-2507',
        'num_labels': num_classes,
    }
}

In [ ]:

# Set up BenchmarkRunner with QLoRA
# 10 epochs is typically sufficient for QLoRA fine-tuning
qwen_runner = BenchmarkRunner(
    model_configs=[qwen_config],
    augmentations=[None],
    task_type="classification",
    device="cuda",  # Use GPU for QLoRA
    epochs=3,  # 3 epochs recommended for QLoRA
    batch_size=1,  # Small batch size for 4B model
    learning_rate=2e-4,  # Standard LoRA learning rate
    use_kfold=False,  # Disable k-fold for faster training
    path_start="difficulty_train"
)

# Train the QLoRA model on recipe text
# BenchmarkRunner.run() handles data splitting internally
print("\nTraining Qwen3-4B with QLoRA on recipe text...")
qwen_runner.run(X_text, y_difficult)


### 4. Evaluation
Evaluate the classification models using an external test dataset.


In [ ]:
# Add necessary imports at the top
from utils.metrics import METRIC_REGISTRY
from utils.utils import RESULTS_DIR
from pipelines_torch.models import HuggingFaceQLoRAWrapper
from pipelines_torch.base import SimplePredictor

# Preprocess test set
test_df = load_csv('recipes_df_test_bis.csv')
test_df = test_df.dropna(subset=['embeddings_class', 'difficult'])
test_df['difficult'] = test_df['difficult'].replace({'A challenge': 'More effort'})
print('Unique difficulty labels after mapping (test):', test_df['difficult'].unique())

# Prepare embeddings for embedding-based models
X_test, y_test = prepare_embeddings_data(test_df, target_column='difficult', embedding_column='embeddings_class')
le_test = LabelEncoderHelper()
le_test.fit(y_test)
y_test_enc = le.transform(y_test)  # Use train label encoder for consistency

unique, counts = np.unique(y_test, return_counts=True)
class_dist = dict(zip(unique, counts))
print("Validation set class distribution:")
for cls, cnt in class_dist.items():
    print(f"Class {cls}: {cnt}")

def evaluate_saved_models_difficulty(
    model_configs: list,
    X: np.ndarray,
    y: np.ndarray,
    path_start: str = 'difficulty_train'
) -> pd.DataFrame:
    """
    Evaluate saved difficulty classification models on a test set.
    
    Args:
        model_configs: List of model configuration dictionaries
        X: Test features (embeddings)
        y: Test labels (encoded)
        path_start: Path prefix for loading saved models
        
    Returns:
        DataFrame with model names and their metric scores
    """
    metric_map = {
        "accuracy": "accuracy",
        "f1_macro": "f1",
        "precision_macro": "precision",
        "recall_macro": "recall",
        "roc_auc": "roc_auc",
        "pr_auc": "pr_auc",
    }
    
    records = []
    
    for model_cfg in model_configs:
        model_name = model_cfg['name']
        
        try:
            # Load trained model
            model = load_model(
                model_cfg['class'],
                model_name,
                model_cfg['params'],
                path_start=path_start,
                augmentation='none'
            )
            
            print(f"Evaluating {model_name}...")
            
            # Create predictor
            predictor = SimplePredictor(
                model=model,
                task_type='classification',
                batch_size=32
            )
            
            # Get probabilities for all metrics
            probs = predictor.predict_proba(X)
            
            # Compute all metrics from the map
            scores = {
                label: float(METRIC_REGISTRY[key](y, probs))
                for label, key in metric_map.items()
            }
            
            records.append({
                "model": model_name,
                **scores,
            })
            print(f"  ✓ {model_name} complete")
            
        except FileNotFoundError:
            print(f"  ⚠️ Skipping {model_name}: checkpoint not found")
            continue
        except Exception as e:
            print(f"  ✗ Error evaluating {model_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    return pd.DataFrame.from_records(records)

# Evaluate all embedding-based models
print("\n" + "="*60)
print("Evaluating embedding-based models...")
print("="*60)
test_results_df = evaluate_saved_models_difficulty(
    model_configs=model_configs,
    X=X_test,
    y=y_test_enc,
    path_start='difficulty_train'
)

# Evaluate HuggingFace QLoRA model from Kaggle input (text-based)
print("\n" + "="*60)
print("Evaluating HuggingFace QLoRA model from Kaggle input...")
print("="*60)

try:
    # Prepare text data for QLoRA model - reconstruct recipe_text column
    # Based on your working code, the text is in 'recipe_text' column
    # If it's not available, we need to reconstruct it from available columns
    
    if 'recipe_text' in test_df.columns:
        X_test_text = test_df['recipe_text'].dropna().tolist()
        print(f"✓ Using existing recipe_text column: {len(X_test_text)} samples")
    elif 'directions' in test_df.columns and 'ingredients' in test_df.columns:
        # Reconstruct recipe_text like in training
        def create_recipe_text(row):
            ingredients = row.get('ingredients', '')
            directions = row.get('directions', '')
            name = row.get('recipe_name', '')
            return f"Recipe: {name}\nIngredients: {ingredients}\nDirections: {directions}"
        
        test_df['recipe_text'] = test_df.apply(create_recipe_text, axis=1)
        X_test_text = test_df['recipe_text'].tolist()
        print(f"✓ Reconstructed recipe_text from ingredients and directions: {len(X_test_text)} samples")
    else:
        raise ValueError(
            "Cannot find recipe_text column in test_df. "
            "Available columns: " + ", ".join(test_df.columns)
        )
    
    print(f"Processing {len(X_test_text)} test samples with QLoRA...")
    
    # Load the QLoRA model
    qlora_params = {
        'model_name': 'Qwen/Qwen3-4B-Instruct-2507',
        'num_labels': 2,
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    }
    
    qlora_model = load_model(
        model_class=HuggingFaceQLoRAWrapper,
        model_name='hf_qlora_classifier',
        params=qlora_params,
        path_start='qlorafood',
        augmentation='none'
    )
    
    # DON'T use SimplePredictor for text models!
    # Call predict_proba directly on the HuggingFace model
    print("Getting predictions from QLoRA model...")
    probs_qlora = qlora_model.predict_proba(X_test_text)
    
    print(f"✓ Generated {len(probs_qlora)} predictions")
    
    # Compute all metrics using the same metric map
    metric_map = {
        "accuracy": "accuracy",
        "f1_macro": "f1",
        "precision_macro": "precision",
        "recall_macro": "recall",
        "roc_auc": "roc_auc",
        "pr_auc": "pr_auc",
    }
    
    qlora_scores = {
        label: float(METRIC_REGISTRY[key](y_test_enc, probs_qlora))
        for label, key in metric_map.items()
    }
    
    qlora_record = {
        "model": "hf_qlora_classifier_kaggle",
        **qlora_scores,
    }
    
    # Append to results
    test_results_df = pd.concat([
        test_results_df,
        pd.DataFrame([qlora_record])
    ], ignore_index=True)
    
    print(f"  ✓ QLoRA model evaluation complete")
    print(f"  Results: {qlora_scores}")
    
except Exception as e:
    print(f"  ✗ Error evaluating QLoRA model: {e}")
    import traceback
    traceback.print_exc()
    print("  Skipping QLoRA model evaluation...")

# Save test results
print("\n" + "="*60)
os.makedirs(RESULTS_DIR, exist_ok=True)
test_results_csv = os.path.join(RESULTS_DIR, "difficulty_test_results.csv")
test_results_df.to_csv(test_results_csv, index=False)
print(f"✓ Test results saved to {test_results_csv}")

# Plot grouped bar chart of metrics for all models
import matplotlib.pyplot as plt

test_results_df.set_index('model').plot(kind='bar', figsize=(12, 6))
plt.title("Difficulty Classification Test Metrics (recipes_df_test_bis.csv)")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Metric", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Sort by accuracy and display
test_results_df.sort_values('accuracy', ascending=False)

In [ ]:
# Preprocess test set
test_df = load_csv('recipes_df_test_bis.csv')
test_df = test_df.dropna(subset=['embeddings_class', 'difficult'])
test_df['difficult'] = test_df['difficult'].replace({'A challenge': 'More effort'})
print('Unique difficulty labels after mapping (test):', test_df['difficult'].unique())
X_test, y_test = prepare_embeddings_data(test_df, target_column='difficult', embedding_column='embeddings_class')
le_test = LabelEncoderHelper(); le_test.fit(y_test)
y_test_enc = le.transform(y_test)  # Use train label encoder for consistency
unique, counts = np.unique(y_test, return_counts=True)
class_dist = dict(zip(unique, counts))
print("Validation set class distribution:")
for cls, cnt in class_dist.items():
    print(f"Class {cls}: {cnt}")
# Evaluate all models
test_results = []
for model_cfg in model_configs:
    model_name = model_cfg['name']
    # Load trained model
    model = load_model(model_cfg['class'], model_name, model_cfg['params'], path_start='difficulty_train2')
    is_torch_model = hasattr(model, 'parameters')
    pipeline = SimplePredictor(
        model=model,
        task_type='classification'
    )
    y_pred = pipeline.predict(X_test)
    # Collect metrics
    result = {}
    for metric in metrics:
        metric_key = getattr(metric, 'name', None) or getattr(metric, '__name__', None) or str(metric)
        result[metric_key] = metric(y_test_enc, y_pred)
    result['model'] = model_name
    test_results.append(result)

# Save test results
test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(os.path.join(RESULTS_DIR, "difficulty_test_results_2.csv"), index=False)
print("Test results saved to", os.path.join(RESULTS_DIR, "difficulty_test_results_2.csv"))

# Plot grouped bar chart of metrics for all models
import matplotlib.pyplot as plt
test_results_df.set_index('model').plot(kind='bar', figsize=(12,6))
plt.title("Difficulty Classification Test Metrics (recipes_df_test.csv)")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=45)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

test_results_df

### 5. Tuning a Xgboost model with Optuna

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import xgboost as xgb
import optuna

# Split your data
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.3, random_state=42)

def objective(trial):
    params = {
        'objective': 'multi:softmax',
        'eval_metric': 'mlogloss',
        'n_estimators': trial.suggest_int('n_estimators', 200, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1, 10),
        'n_jobs': -1,
        'num_class': len(np.unique(y_train)),
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    score = f1_score(y_test, y_pred, average='macro')
    return score

# 2. Create and run the Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) # n_trials can be increased for a more thorough search

# 3. Get and print the best parameters
best_xgb_params = study.best_trial.params
print("Best trial score (F1 Macro):", study.best_value)
print("Best trial parameters:")
print(best_xgb_params)

In [ ]:
# Get the xgboost model config
xgb_config = next((config for config in model_configs if config['name'] == 'xgboost_classifier'), None)
if xgb_config:
    model_name = "xgboost_classifier_tuned"
    print(f"--- Training {model_name} with best parameters using BenchmarkRunner ---")

    # Create a new model config with tuned parameters
    tuned_model_config = {
        'name': model_name,
        'class': xgb_config['class'],
        'params': best_xgb_params
    }

    # Use BenchmarkRunner with the tuned model
    tuned_runner = BenchmarkRunner(
        model_configs=[tuned_model_config],
        augmentations=[None],  # Can use best_augmentation if desired
        metrics=metrics,
        task_type="classification",
        device="cpu",
        epochs=150,
        batch_size=32,
        use_kfold=False,
        learning_rate=1e-4,
        path_start="difficulty_train",
        random_state=42,
    )
    
    # Run training - results are automatically saved to results/difficulty_train/
    tuned_runner.run(X, y_enc)
    
    print(f"--- Finished training and saved model: {model_name} ---")
    print("Model and metrics saved to results/difficulty_train/")
else:
    print("XGBoost configuration not found in model_configs.")

In [ ]:
# Load the test data (reusing preprocessing from your notebook)
test_df = load_csv('recipes_df_test_bis.csv')
test_df = test_df.dropna(subset=['embeddings_class', 'difficult'])
test_df['difficult'] = test_df['difficult'].replace({'A challenge': 'More effort'})
X_test, y_test = prepare_embeddings_data(test_df, target_column='difficult', embedding_column='embeddings_class')
y_test_enc = le.transform(y_test)  # Use the label encoder fitted on the training data

# Define the metric map for evaluation
metric_map = {
    "accuracy": "accuracy",
    "f1_macro": "f1",
    "precision_macro": "precision",
    "recall_macro": "recall",
    "roc_auc": "roc_auc",
    "pr_auc": "pr_auc",
}

try:
    # Load the tuned model
    xgb_config = next((config for config in model_configs if config['name'] == 'xgboost_classifier'), None)
    tuned_model = load_model(
        xgb_config['class'], 
        model_name, 
        best_xgb_params, 
        path_start='difficulty_train',
        augmentation='none'
    )

    print(f"Evaluating {model_name}...")
    
    # Create predictor
    predictor = SimplePredictor(
        model=tuned_model,
        task_type='classification',
        batch_size=32
    )
    
    # Get probabilities for all metrics
    probs = predictor.predict_proba(X_test)
    
    # Compute all metrics from the map
    tuned_results = {
        label: float(METRIC_REGISTRY[key](y_test_enc, probs))
        for label, key in metric_map.items()
    }
    tuned_results['model'] = model_name
    
    tuned_results_df = pd.DataFrame([tuned_results])
    print(f"  ✓ {model_name} evaluation complete")
    
except Exception as e:
    print(f"  ✗ Error evaluating {model_name}: {e}")
    import traceback
    traceback.print_exc()
    tuned_results_df = pd.DataFrame()

tuned_results_df

## Meal Type Classification
### 1. Data Preprocessing
Load and preprocess data for meal type classification.

In [ ]:
meal_types = ['Lunch recipes', 'Dinner recipes', 'Breakfast recipes']
meal_df = filter_meal_types(load_csv('recipes_df.csv'), meal_types)
if 'embeddings_class' in meal_df.columns and not meal_df['embeddings_class'].isnull().any():
    X_meal, y_meal = prepare_embeddings_data(meal_df, target_column='meal_type', embedding_column='embeddings_class')
    le_meal = LabelEncoderHelper(); le_meal.fit(y_meal)
    y_meal_enc = le_meal.transform(y_meal)
    X_train_meal, X_val_meal, _, y_train_meal, y_val_meal, _ = train_val_test_split(X_meal, y_meal_enc, val_size=0.15, test_size=0, stratify=y_meal_enc)
    input_dim = X_train_meal.shape[1]
    num_classes = len(np.unique(y_train_meal))
    model_names = [
        'mlp_classifier', 'deep_mlp_classifier', 'random_forest_classifier', 'xgboost_classifier', 'lightgbm_classifier'
    ]
    model_configs = [
        {
            'name': name,
            'class': MODEL_REGISTRY[name],
            'params': {'input_dim': input_dim, 'num_classes': num_classes} if 'mlp' in name or 'transformer' in name else {}
        }
        for name in model_names
    ]
    # Define metrics for meal type classification
    metrics = [METRIC_REGISTRY['f1'], METRIC_REGISTRY['recall'], METRIC_REGISTRY['precision'], METRIC_REGISTRY['accuracy']]
    unique, counts = np.unique(y_val_meal, return_counts=True)
    class_dist = dict(zip(unique, counts))
    print("Validation set class distribution:")
    for cls, cnt in class_dist.items():
        print(f"Class {cls}: {cnt}")

### 2. Data augmentation
Since the number of instances in the dataset is limited, I need to do LLM data augmentation in order to train the model.

In [ ]:
# --- Augment 50 samples per class using 5 methods ---
from data_augmentation.text_aug import TEXT_AUGMENTATION_REGISTRY
import random

API_KEY = os.getenv("GOOGLE_API_KEY")
aug_methods = [
    "llm_paraphrase",
    "llm_synonym",
    "llm_style",
    "classical_synonym",
    "classical_mixed"
]
N_PER_CLASS = 50

augmented_texts = []
augmented_labels = []
for class_id in np.unique(y_train_meal):
    class_name = le_meal.inverse_transform([class_id])[0]
    class_texts = meal_df[meal_df['meal_type'] == class_name]['recipe_text'].tolist()
    # If not enough, sample with replacement
    if len(class_texts) < N_PER_CLASS:
        class_texts = random.choices(class_texts, k=N_PER_CLASS)
    else:
        class_texts = random.sample(class_texts, N_PER_CLASS)
    per_method = max(1, N_PER_CLASS // len(aug_methods))
    class_augmented = []
    for aug_method in aug_methods:
        if aug_method.startswith("llm_"):
            augmented = TEXT_AUGMENTATION_REGISTRY[aug_method](class_texts, api_key=API_KEY)
        else:
            augmented = TEXT_AUGMENTATION_REGISTRY[aug_method](class_texts)
        flat_aug = [x[0] for x in augmented if x and x[0]]
        class_augmented.extend(random.sample(flat_aug, min(per_method, len(flat_aug))))
    class_augmented = class_augmented[:N_PER_CLASS]
    augmented_texts.extend(class_augmented)
    augmented_labels.extend([class_id] * len(class_augmented))

print(f"Generated {len(augmented_texts)} augmented samples ({N_PER_CLASS} per class).")
for i in range(min(5, len(augmented_texts))):
    print(f"Augmented sample {i+1}: {augmented_texts[i]} (label={augmented_labels[i]})")

In [ ]:
# --- Embed augmented samples and add to training set ---
from google import genai
from google.genai import types
from google.api_core import retry

is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

@retry.Retry(predicate=is_retriable, timeout=600.0)
def embed_fn_class(text: str) -> list[float]:
    client = genai.Client(api_key=GOOGLE_API_KEY)
    response = client.models.embed_content(
        model="models/text-embedding-004",
        contents=text,
        config=types.EmbedContentConfig(
            task_type="classification",
        ),
    )
    return response.embeddings[0].values

aug_embeds = []
for text in augmented_texts:
    try:
        emb = embed_fn_class(text)
        aug_embeds.append(emb)
    except Exception as e:
        print(f"Embedding failed for: {text[:30]}... Error: {e}")

if len(aug_embeds) == len(augmented_labels):
    import numpy as np
    X_train_meal = np.vstack([X_train_meal, np.array(aug_embeds)])
    y_train_meal = np.concatenate([y_train_meal, np.array(augmented_labels)])
    print(f"Added {len(aug_embeds)} embedded augmented samples to training set, distributed per class.")
else:
    print("Some embeddings failed, not all augmented samples were added to training set.")

In [ ]:
import os
np.save(os.path.join(RESULTS_DIR, 'meal_data/X_train_meal.npy'), X_train_meal)
np.save(os.path.join(RESULTS_DIR, 'meal_data/y_train_meal.npy'), y_train_meal)
np.save(os.path.join(RESULTS_DIR, 'meal_data/X_val_meal.npy'), X_val_meal)
np.save(os.path.join(RESULTS_DIR, 'meal_data/y_val_meal.npy'), y_val_meal)

### 3. Training & Testing
Train and test all models using the augmented training set.

In [ ]:
# Load meal type data
X_train_meal = np.load(os.path.join(RESULTS_DIR, 'meal_data/X_train_meal.npy'))
y_train_meal = np.load(os.path.join(RESULTS_DIR, 'meal_data/y_train_meal.npy'))
X_val_meal = np.load(os.path.join(RESULTS_DIR, 'meal_data/X_val_meal.npy'))
y_val_meal = np.load(os.path.join(RESULTS_DIR, 'meal_data/y_val_meal.npy'))

# Combine train and val for BenchmarkRunner (it will split internally)
X_meal_combined = np.vstack([X_train_meal, X_val_meal])
y_meal_combined = np.concatenate([y_train_meal, y_val_meal])

print(f"Combined dataset size: {len(X_meal_combined)} samples")
print(f"Training models for meal type classification...")

# Use BenchmarkRunner for training
meal_runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],
    metrics=metrics,
    task_type="classification",
    device="cpu",
    epochs=800,
    batch_size=32,
    early_stopping=40,
    use_kfold=False,
    use_class_weights=True,
    learning_rate=2e-4,
    weight_decay=1e-4,
    path_start="meal_train",
    random_state=42,
)

# Run training - results are automatically saved to results/meal_train/
meal_runner.run(X_meal_combined, y_meal_combined)

print("\nMeal type training complete! Models and metrics saved to results/meal_train/")



### 5. Evaluation
Generate plots and metrics for all models.


In [ ]:
# Preprocess test set
test_df = load_csv('recipes_df_test.csv')

# Create meal_type column based on dish_type
def extract_meal_type(dish_type):
    dish_type_lower = str(dish_type).lower()
    if 'lunch' in dish_type_lower:
        return 'Lunch recipes'
    elif 'dinner' in dish_type_lower:
        return 'Dinner recipes'
    elif 'breakfast' in dish_type_lower:
        return 'Breakfast recipes'
    else:
        return None

test_df['meal_type'] = test_df['dish_type'].apply(extract_meal_type)

meal_types = ['Lunch recipes', 'Dinner recipes', 'Breakfast recipes']
test_meal_df = test_df[test_df['meal_type'].isin(meal_types)]
test_meal_df = test_meal_df.dropna(subset=['embeddings_class', 'meal_type'])
print('Unique meal type labels (test):', test_meal_df['meal_type'].unique())
X_test, y_test = prepare_embeddings_data(test_meal_df, target_column='meal_type', embedding_column='embeddings_class')
le_test = LabelEncoderHelper(); 
le_test.fit(y_test)
y_test_enc = le_meal.transform(y_test)  # Use train label encoder for consistency
unique, counts = np.unique(y_test, return_counts=True)
class_dist = dict(zip(unique, counts))
print("Test set class distribution:")
for cls, cnt in class_dist.items():
    print(f"Class {cls}: {cnt}")

def evaluate_saved_models_meal(
    model_configs: list,
    X: np.ndarray,
    y: np.ndarray,
    path_start: str = 'meal_train'
) -> pd.DataFrame:
    """
    Evaluate saved meal type classification models on a test set.
    
    Args:
        model_configs: List of model configuration dictionaries
        X: Test features (embeddings)
        y: Test labels (encoded)
        path_start: Path prefix for loading saved models
        
    Returns:
        DataFrame with model names and their metric scores
    """
    metric_map = {
        "accuracy": "accuracy",
        "f1_macro": "f1",
        "precision_macro": "precision",
        "recall_macro": "recall",
    }
    
    records = []
    
    for model_cfg in model_configs:
        model_name = model_cfg['name']
        
        try:
            # Load trained model
            model = load_model(
                model_cfg['class'],
                model_name,
                model_cfg['params'],
                path_start=path_start,
                augmentation='none'
            )
            
            print(f"Evaluating {model_name}...")
            
            # Create predictor
            predictor = SimplePredictor(
                model=model,
                task_type='classification',
                batch_size=32
            )
            
            # Get probabilities for all metrics
            probs = predictor.predict_proba(X)
            
            # Compute all metrics from the map
            scores = {
                label: float(METRIC_REGISTRY[key](y, probs))
                for label, key in metric_map.items()
            }
            
            records.append({
                "model": model_name,
                **scores,
            })
            print(f"  ✓ {model_name} complete")
            
        except FileNotFoundError:
            print(f"  ⚠️ Skipping {model_name}: checkpoint not found")
            continue
        except Exception as e:
            print(f"  ✗ Error evaluating {model_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    return pd.DataFrame.from_records(records)

# Evaluate all models
print("\n" + "="*60)
print("Evaluating meal type classification models...")
print("="*60)
test_results_df = evaluate_saved_models_meal(
    model_configs=model_configs,
    X=X_test,
    y=y_test_enc,
    path_start='meal_train'
)

# Save test results
os.makedirs(RESULTS_DIR, exist_ok=True)
test_results_csv = os.path.join(RESULTS_DIR, "meal_type_test_results.csv")
test_results_df.to_csv(test_results_csv, index=False)
print(f"✓ Test results saved to {test_results_csv}")

# Plot grouped bar chart of metrics for all models
import matplotlib.pyplot as plt
test_results_df.set_index('model').plot(kind='bar', figsize=(12,6))
plt.title("Meal Type Classification Test Metrics (recipes_df_test.csv)")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Metric", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Sort by accuracy and display
test_results_df.sort_values('accuracy', ascending=False)

In [ ]:
from sklearn.metrics import classification_report
from utils.visualization import plot_per_class_metrics

# 1. Find the best model based on F1-score
best_model_name = test_results_df.loc[test_results_df['f1_score'].idxmax()]['model']
print(f"\n--- Per-Class Analysis for Best Model: {best_model_name} ---")

# 2. Get predictions for the best model again
best_model_cfg = next(cfg for cfg in model_configs if cfg['name'] == best_model_name)
model = load_model(best_model_cfg['class'], best_model_name, best_model_cfg['params'], path_start='meal_train')

is_torch_model = hasattr(model, 'parameters')
pipeline = SimplePredictor(
    model=model,
    task_type='classification'
)
y_pred_best = pipeline.predict(X_test)

# 3. Generate and plot per-class metrics
class_names = le_test.classes()
report = classification_report(y_test_enc, y_pred_best, target_names=class_names, output_dict=True)
report_df = pd.DataFrame(report).transpose()

print("\nClassification Report:")
print(report_df)

plot_per_class_metrics(report_df, title=f"Per-Class Metrics for {best_model_name}")

# 4. Generate and plot one-vs-rest confusion matrices for each class
print("\nOne-vs-Rest Confusion Matrices:")
for i, class_name in enumerate(class_names):
    # Binarize true and predicted labels for the current class
    y_true_ovr = (y_test_enc == i).astype(int)
    y_pred_ovr = (y_pred_best == i).astype(int)
    
    plot_confusion_matrix(
        y_true_ovr, 
        y_pred_ovr, 
        labels=[1, 0],  # Positive class (1) first
        title=f"Confusion Matrix for '{class_name}' vs. Rest"
    )

In [ ]:
# --- New: Breakfast vs. Non-Breakfast Analysis ---
print("\n--- Analysis: Breakfast vs. Non-Breakfast ---")

# 1. Identify the integer label for 'Breakfast recipes'
try:
    # Convert to list just in case it's not
    class_names_list = list(class_names)
    breakfast_idx = class_names_list.index('Breakfast recipes')
except ValueError:
    print("Error: 'Breakfast recipes' not found in class labels.")
else:
    # 2. Create binary labels: 1 for Breakfast, 0 for Non-Breakfast (Lunch/Dinner)
    y_true_binary = (y_test_enc == breakfast_idx).astype(int)
    y_pred_binary = (y_pred_best == breakfast_idx).astype(int)
    
    binary_target_names = ['Non-Breakfast', 'Breakfast'] # 0, 1
    
    # 3. Generate and print the binary classification report
    binary_report = classification_report(
        y_true_binary, 
        y_pred_binary, 
        target_names=binary_target_names, 
        output_dict=True
    )
    binary_report_df = pd.DataFrame(binary_report).transpose()
    
    # Filter to show only the per-class metrics and make a copy
    binary_class_report_df = binary_report_df.loc[binary_target_names].copy()
    
    # Add the overall accuracy as a new column
    binary_class_report_df['accuracy'] = binary_report['accuracy']
    
binary_class_report_df


## Nutrient Value Prediction (Multi-output Regression)
### 1. Data Preprocessing
Load and preprocess data for nutrient regression.


In [ ]:
import ast
nutrient_df = load_csv('recipes_df.csv')
nutrient_keys = set()
for val in nutrient_df['nutrients'].dropna():
    try:
        d = ast.literal_eval(val) if isinstance(val, str) else val
        if isinstance(d, dict):
            nutrient_keys.update(d.keys())
    except Exception:
        continue
nutrient_keys = sorted([k for k in nutrient_keys if k])
def parse_nutrients(row):
    try:
        d = ast.literal_eval(row) if isinstance(row, str) else row
        if not isinstance(d, dict):
            return [None]*len(nutrient_keys)
        vals = []
        for k in nutrient_keys:
            v = d.get(k, None)
            if v is None or v == '':
                return [None]*len(nutrient_keys)
            v = v.replace('g','').replace('kcal','').strip()
            try:
                vals.append(float(v))
            except Exception:
                return [None]*len(nutrient_keys)
        return vals
    except Exception:
        return [None]*len(nutrient_keys)
nutrient_df['nutrient_vec'] = nutrient_df['nutrients'].apply(parse_nutrients)
valid_nutrient_df = nutrient_df[nutrient_df['nutrient_vec'].apply(lambda x: None not in x)]
if 'embeddings_reg' in valid_nutrient_df.columns and not valid_nutrient_df['embeddings_reg'].isnull().any():
    X_nutr, y_nutr = prepare_embeddings_data(valid_nutrient_df, target_column='nutrient_vec', embedding_column='embeddings_reg')
    X_train_nutr, X_val_nutr, _, y_train_nutr, y_val_nutr, _ = train_val_test_split(X_nutr, np.vstack(y_nutr), val_size=0.15, test_size=0, stratify=None)
    input_dim = X_train_nutr.shape[1]
    output_dim = y_train_nutr.shape[1]
    print(f"Nutrient data prepared: {len(X_nutr)} samples, {input_dim} input features, {output_dim} output nutrients")


### 2. Model Setup
Define model configurations and metrics for nutrient regression.


In [ ]:
model_names = [
    'mlp_regressor', 
    'deep_mlp_regressor', 
    'random_forest_regressor', 
    'xgboost_regressor', 
    'lightgbm_regressor'
]
model_configs = [
    {
        'name': name,
        'class': MODEL_REGISTRY[name],
        'params': {'input_dim': input_dim, 'output_dim': output_dim} if 'mlp' in name else {}
    }
    for name in model_names
]
metrics = [METRIC_REGISTRY['mse'], METRIC_REGISTRY['mae'], METRIC_REGISTRY['r2']]


### 3. Training & Testing
Train and test all models for nutrient regression.


In [ ]:
# Train nutrient regression models using BenchmarkRunner
print(f"Training nutrient regression models on {len(X_nutr)} samples...")

nutrient_runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],
    metrics=metrics,
    task_type="regression",
    device="cpu",
    epochs=2000,
    batch_size=32,
    early_stopping=30,
    use_kfold=False,
    learning_rate=1e-4,
    weight_decay=1e-5,
    path_start="nutrient_train",
    random_state=42,
)

# Run training - results are automatically saved to results/nutrient_train/
nutrient_runner.run(X_nutr, np.vstack(y_nutr))

print("\nNutrient regression training complete! Models and metrics saved to results/nutrient_train/")



### 4. Evaluation
Generate plots and metrics for all models.


In [ ]:
# --- 4. Evaluation on External Dataset ---
import matplotlib.pyplot as plt

# --- Preprocess Test Set ---
test_results_path = os.path.join(RESULTS_DIR, "nutrient_test_results.csv")

if os.path.exists(test_results_path):
    print(f"Test results already exist at {test_results_path}, loading from file.")
    test_results_df = pd.read_csv(test_results_path)
    # Reload test data for detailed analysis later
    test_df = load_csv('recipes_df_test.csv')
    test_df = test_df.dropna(subset=['embeddings_reg', 'nutrients'])
    test_df['nutrient_vec'] = test_df['nutrients'].apply(parse_nutrients)
    valid_test_df = test_df[test_df['nutrient_vec'].apply(lambda x: None not in x)]
    X_test, y_test_list = prepare_embeddings_data(valid_test_df, target_column='nutrient_vec', embedding_column='embeddings_reg')
    y_test = np.vstack(y_test_list)
else:
    print("Evaluating models on external recipes_df_test.csv dataset...")
    test_df = load_csv('recipes_df_test.csv')
    test_df = test_df.dropna(subset=['embeddings_reg', 'nutrients'])
    
    test_df['nutrient_vec'] = test_df['nutrients'].apply(parse_nutrients)
    valid_test_df = test_df[test_df['nutrient_vec'].apply(lambda x: None not in x)]
    
    X_test, y_test_list = prepare_embeddings_data(valid_test_df, target_column='nutrient_vec', embedding_column='embeddings_reg')
    y_test = np.vstack(y_test_list)

    def evaluate_saved_models_nutrient(
        model_configs: list,
        X: np.ndarray,
        y: np.ndarray,
        path_start: str = 'nutrient_train'
    ) -> pd.DataFrame:
        """
        Evaluate saved nutrient regression models on a test set.
        
        Args:
            model_configs: List of model configuration dictionaries
            X: Test features (embeddings)
            y: Test labels (nutrient values)
            path_start: Path prefix for loading saved models
            
        Returns:
            DataFrame with model names and their metric scores
        """
        metric_map = {
            "mean_squared_error": "mse",
            "mean_absolute_error": "mae",
            "r2_score": "r2",
        }
        
        records = []
        
        for model_cfg in model_configs:
            model_name = model_cfg['name']
            
            try:
                # Load trained model
                model = load_model(
                    model_cfg['class'],
                    model_name,
                    model_cfg['params'],
                    path_start=path_start,
                    augmentation='none'
                )
                
                print(f"Evaluating {model_name}...")
                
                # Create predictor
                predictor = SimplePredictor(
                    model=model,
                    task_type='regression',
                    batch_size=32
                )
                
                # Get predictions
                y_pred = predictor.predict(X)
                
                # Compute all metrics from the map
                scores = {
                    label: float(METRIC_REGISTRY[key](y, y_pred))
                    for label, key in metric_map.items()
                }
                
                records.append({
                    "model": model_name,
                    **scores,
                })
                print(f"  ✓ {model_name} complete")
                
            except FileNotFoundError:
                print(f"  ⚠️ Skipping {model_name}: checkpoint not found")
                continue
            except Exception as e:
                print(f"  ✗ Error evaluating {model_name}: {e}")
                import traceback
                traceback.print_exc()
                continue
        
        return pd.DataFrame.from_records(records)

    # --- Evaluate all models ---
    test_results_df = evaluate_saved_models_nutrient(
        model_configs=model_configs,
        X=X_test,
        y=y_test,
        path_start='nutrient_train'
    )

    # Save test results
    test_results_df.to_csv(test_results_path, index=False)
    print("Test results saved to", test_results_path)

# --- Visualize Results ---
# Plot grouped bar chart of metrics for all models
fig, ax1 = plt.subplots(figsize=(12, 6))
print(test_results_df.columns)
test_results_df.set_index('model')[['mean_squared_error', 'mean_absolute_error']].plot(kind='bar', ax=ax1)
ax1.set_ylabel('Error (MSE, MAE)')
ax1.set_xlabel('Model')
ax1.set_title("Nutrient Regression Test Metrics (recipes_df_test.csv)")
ax1.tick_params(axis='x', rotation=45)

# Plot R2 score on a secondary y-axis
ax2 = ax1.twinx()
test_results_df.set_index('model')['r2_score'].plot(kind='line', marker='o', color='r', ax=ax2)
ax2.set_ylabel('R² Score', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax2.set_ylim(0, max(1.0, test_results_df['r2_score'].max() * 1.1)) # Adjust ylim dynamically

fig.tight_layout()
plt.show()

test_results_df

In [ ]:
# --- 4. Evaluation on External Dataset ---
import matplotlib.pyplot as plt

# --- Preprocess Test Set ---
test_results_path = os.path.join(RESULTS_DIR, "nutrient_test_results2.csv")

if os.path.exists(test_results_path):
    print(f"Test results already exist at {test_results_path}, loading from file.")
    test_results_df2 = pd.read_csv(test_results_path)
    # Reload test data for detailed analysis later
    test_df = load_csv('recipes_df_test_bis.csv')
    test_df = test_df.dropna(subset=['embeddings_reg', 'nutrients'])
    test_df['nutrient_vec'] = test_df['nutrients'].apply(parse_nutrients)
    valid_test_df = test_df[test_df['nutrient_vec'].apply(lambda x: None not in x)]
    X_test, y_test_list = prepare_embeddings_data(valid_test_df, target_column='nutrient_vec', embedding_column='embeddings_reg')
    y_test = np.vstack(y_test_list)
else:
    print("Evaluating models on external recipes_df_test_bis.csv dataset...")
    test_df = load_csv('recipes_df_test_bis.csv')
    test_df = test_df.dropna(subset=['embeddings_reg', 'nutrients'])
    
    test_df['nutrient_vec'] = test_df['nutrients'].apply(parse_nutrients)
    valid_test_df = test_df[test_df['nutrient_vec'].apply(lambda x: None not in x)]
    
    X_test, y_test_list = prepare_embeddings_data(valid_test_df, target_column='nutrient_vec', embedding_column='embeddings_reg')
    y_test = np.vstack(y_test_list)

    def evaluate_saved_models_nutrient(
        model_configs: list,
        X: np.ndarray,
        y: np.ndarray,
        path_start: str = 'nutrient_train'
    ) -> pd.DataFrame:
        """
        Evaluate saved nutrient regression models on a test set.
        
        Args:
            model_configs: List of model configuration dictionaries
            X: Test features (embeddings)
            y: Test labels (nutrient values)
            path_start: Path prefix for loading saved models
            
        Returns:
            DataFrame with model names and their metric scores
        """
        metric_map = {
            "mean_squared_error": "mse",
            "mean_absolute_error": "mae",
            "r2_score": "r2",
        }
        
        records = []
        
        for model_cfg in model_configs:
            model_name = model_cfg['name']
            
            try:
                # Load trained model
                model = load_model(
                    model_cfg['class'],
                    model_name,
                    model_cfg['params'],
                    path_start=path_start,
                    augmentation='none'
                )
                
                print(f"Evaluating {model_name}...")
                
                # Create predictor
                predictor = SimplePredictor(
                    model=model,
                    task_type='regression',
                    batch_size=32
                )
                
                # Get predictions
                y_pred = predictor.predict(X)
                
                # Compute all metrics from the map
                scores = {
                    label: float(METRIC_REGISTRY[key](y, y_pred))
                    for label, key in metric_map.items()
                }
                
                records.append({
                    "model": model_name,
                    **scores,
                })
                print(f"  ✓ {model_name} complete")
                
            except FileNotFoundError:
                print(f"  ⚠️ Skipping {model_name}: checkpoint not found")
                continue
            except Exception as e:
                print(f"  ✗ Error evaluating {model_name}: {e}")
                import traceback
                traceback.print_exc()
                continue
        
        return pd.DataFrame.from_records(records)

    # --- Evaluate all models ---
    test_results_df2 = evaluate_saved_models_nutrient(
        model_configs=model_configs,
        X=X_test,
        y=y_test,
        path_start='nutrient_train'
    )

    # Save test results
    test_results_df2.to_csv(test_results_path, index=False)
    print("Test results saved to", test_results_path)

# --- Visualize Results ---
# Plot grouped bar chart of metrics for all models
fig, ax1 = plt.subplots(figsize=(12, 6))
print(test_results_df2.columns)
test_results_df2.set_index('model')[['mean_squared_error', 'mean_absolute_error']].plot(kind='bar', ax=ax1)
ax1.set_ylabel('Error (MSE, MAE)')
ax1.set_xlabel('Model')
ax1.set_title("Nutrient Regression Test Metrics (recipes_df_test_bis.csv)")
ax1.tick_params(axis='x', rotation=45)

# Plot R2 score on a secondary y-axis
ax2 = ax1.twinx()
test_results_df2.set_index('model')['r2_score'].plot(kind='line', marker='o', color='r', ax=ax2)
ax2.set_ylabel('R² Score', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax2.set_ylim(0, max(1.0, test_results_df2['r2_score'].max() * 1.1)) # Adjust ylim dynamically

fig.tight_layout()
plt.show()

test_results_df2

In [ ]:
# --- Detailed Analysis for Best Model ---
# Find the best model based on R2 score
best_model_name = test_results_df.loc[test_results_df['r2_score'].idxmax()]['model']
print(f"\n--- Detailed Analysis for Best Model: {best_model_name} ---")

# Get predictions for the best model again
best_model_cfg = next(cfg for cfg in model_configs if cfg['name'] == best_model_name)
model = load_model(best_model_cfg['class'], best_model_name, best_model_cfg['params'], path_start='nutrient_train')

test_df = load_csv('recipes_df_test.csv')
test_df = test_df.dropna(subset=['embeddings_reg', 'nutrients'])

test_df['nutrient_vec'] = test_df['nutrients'].apply(parse_nutrients)
valid_test_df = test_df[test_df['nutrient_vec'].apply(lambda x: None not in x)]

X_test, y_test_list = prepare_embeddings_data(valid_test_df, target_column='nutrient_vec', embedding_column='embeddings_reg')
y_test = np.vstack(y_test_list)

is_torch_model = hasattr(model, 'parameters')
pipeline = SimplePredictor(
    model=model,
    task_type='regression',
)
y_pred_best = pipeline.predict(X_test)

# Plot regression results for each nutrient
for i, nutrient_name in enumerate(nutrient_keys):
    if nutrient_name.lower() == "carbs":
        # Remove outliers where carbs > 1000
        mask = y_test[:, i] <= 1000
        plot_regression_results(
            y_test[mask, i],
            y_pred_best[mask, i],
            title=f"'{nutrient_name}' Prediction vs. Actual for {best_model_name} (carbs ≤ 1000)"
        )
    else:
        plot_regression_results(
            y_test[:, i], 
            y_pred_best[:, i], 
            title=f"'{nutrient_name}' Prediction vs. Actual for {best_model_name}"
        )


## Time Prediction (Single-output Regression)
### 1. Data Preprocessing
Load and preprocess data for total time regression.


In [ ]:
time_df = load_csv('recipes_df.csv')
def parse_time(row):
    import ast
    try:
        d = ast.literal_eval(row) if isinstance(row, str) else row
        if not isinstance(d, dict):
            return [None, None]
        prep = d.get('Preparation', '0').replace('No Time','0').replace('mins','').replace('min','').strip()
        cook = d.get('Cooking', '0').replace('No Time','0').replace('mins','').replace('min','').strip()
        try:
            prep = int(round(float(prep)))
        except Exception:
            prep = 0
        try:
            cook = int(round(float(cook)))
        except Exception:
            cook = 0
        return [prep, cook]
    except Exception:
        return [None, None]
time_df['time_vec'] = time_df['times'].apply(parse_time)
time_df['total_time'] = time_df['time_vec'].apply(lambda x: sum(x) if None not in x else None)
valid_time_df = time_df[time_df['time_vec'].apply(lambda x: None not in x) & time_df['total_time'].notnull()]
X_total_time, y_total_time = prepare_embeddings_data(valid_time_df, target_column='total_time', embedding_column='embeddings_reg')
X_train_total, X_val_total, _, y_train_total, y_val_total, _ = train_val_test_split(X_total_time, np.vstack(y_total_time), val_size=0.15, test_size=0, stratify=None)
input_dim = X_train_total.shape[1]
output_dim = y_train_total.shape[1]
print(f"Total time data prepared: {len(X_total_time)} samples, {input_dim} input features")

### 2. Model Setup
Define model configurations and metrics for total time regression.

In [ ]:
model_names = [
    'mlp_regressor', 
    'deep_mlp_regressor', 
    'random_forest_regressor', 
    'xgboost_regressor', 
    'lightgbm_regressor'
]
model_configs = [
    {
        'name': name,
        'class': MODEL_REGISTRY[name],
        'params': {'input_dim': input_dim, 'output_dim': output_dim} if 'mlp' in name else {}
    }
    for name in model_names
]
metrics = [METRIC_REGISTRY['mse'], METRIC_REGISTRY['mae'], METRIC_REGISTRY['r2']]

### 3. Training & Testing
Train and test all models for total time regression.


In [ ]:
# Train total time regression models using BenchmarkRunner
print(f"Training total time regression models on {len(X_total_time)} samples...")

total_time_runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],
    metrics=metrics,
    task_type="regression",
    device="cpu",
    epochs=600,
    batch_size=32,
    early_stopping=20,
    use_kfold=False,
    learning_rate=1e-4,
    weight_decay=1e-4,
    path_start="total_time_train",
    random_state=42,
)

# Prepare target appropriately
y_total_time_prepared = np.vstack(y_total_time).flatten()

# Run training - results are automatically saved to results/total_time_train/
total_time_runner.run(X_total_time, y_total_time_prepared)

print("\nTotal time regression training complete! Models and metrics saved to results/total_time_train/")



### 4. Evaluation
Generate plots and metrics for all models.


In [ ]:
# --- 4. Evaluation on External Dataset (Total Time Regression) ---
import matplotlib.pyplot as plt

# --- Preprocess Test Set ---
test_results_path = os.path.join(RESULTS_DIR, "total_time_test_results.csv")

if os.path.exists(test_results_path):
    print(f"Test results already exist at {test_results_path}, loading from file.")
    test_results_df = pd.read_csv(test_results_path)
    # Reload test data for detailed analysis later
    test_df = load_csv('recipes_df_test.csv')
    test_df = test_df.dropna(subset=['embeddings_reg', 'times'])
    test_df['time_vec'] = test_df['times'].apply(parse_time)
    test_df['total_time'] = test_df['time_vec'].apply(lambda x: sum(x) if None not in x else None)
    valid_test_df = test_df[test_df['total_time'].notnull()]
    X_test, y_test_list = prepare_embeddings_data(valid_test_df, target_column='total_time', embedding_column='embeddings_reg')
    y_test = np.array(y_test_list).flatten()
else:
    print("Evaluating models on external recipes_df_test.csv dataset for total time prediction...")
    test_df = load_csv('recipes_df_test.csv')
    test_df = test_df.dropna(subset=['embeddings_reg', 'times'])
    test_df['time_vec'] = test_df['times'].apply(parse_time)
    test_df['total_time'] = test_df['time_vec'].apply(lambda x: sum(x) if None not in x else None)
    valid_test_df = test_df[test_df['total_time'].notnull()]
    X_test, y_test_list = prepare_embeddings_data(valid_test_df, target_column='total_time', embedding_column='embeddings_reg')
    y_test = np.array(y_test_list).flatten()

    def evaluate_saved_models_time(
        model_configs: list,
        X: np.ndarray,
        y: np.ndarray,
        path_start: str = 'total_time_train'
    ) -> pd.DataFrame:
        """
        Evaluate saved total time regression models on a test set.
        
        Args:
            model_configs: List of model configuration dictionaries
            X: Test features (embeddings)
            y: Test labels (total time values)
            path_start: Path prefix for loading saved models
            
        Returns:
            DataFrame with model names and their metric scores
        """
        metric_map = {
            "mean_squared_error": "mse",
            "mean_absolute_error": "mae",
            "r2_score": "r2",
        }
        
        records = []
        
        for model_cfg in model_configs:
            model_name = model_cfg['name']
            
            try:
                # Load trained model
                model = load_model(
                    model_cfg['class'],
                    model_name,
                    model_cfg['params'],
                    path_start=path_start,
                    augmentation='none'
                )
                
                print(f"Evaluating {model_name}...")
                
                # Create predictor
                predictor = SimplePredictor(
                    model=model,
                    task_type='regression',
                    batch_size=32
                )
                
                # Get predictions
                y_pred = predictor.predict(X)
                
                # Compute all metrics from the map
                scores = {
                    label: float(METRIC_REGISTRY[key](y, y_pred))
                    for label, key in metric_map.items()
                }
                
                records.append({
                    "model": model_name,
                    **scores,
                })
                print(f"  ✓ {model_name} complete")
                
            except FileNotFoundError:
                print(f"  ⚠️ Skipping {model_name}: checkpoint not found")
                continue
            except Exception as e:
                print(f"  ✗ Error evaluating {model_name}: {e}")
                import traceback
                traceback.print_exc()
                continue
        
        return pd.DataFrame.from_records(records)

    # --- Evaluate all models ---
    test_results_df = evaluate_saved_models_time(
        model_configs=model_configs,
        X=X_test,
        y=y_test,
        path_start='total_time_train'
    )

    # Save test results
    test_results_df.to_csv(test_results_path, index=False)
    print("Test results saved to", test_results_path)

# --- Visualize Results ---
fig, ax1 = plt.subplots(figsize=(12, 6))
print(test_results_df.columns)
test_results_df.set_index('model')[['mean_squared_error', 'mean_absolute_error']].plot(kind='bar', ax=ax1)
ax1.set_ylabel('Error (MSE, MAE)')
ax1.set_xlabel('Model')
ax1.set_title("Total Time Regression Test Metrics (recipes_df_test.csv)")
ax1.tick_params(axis='x', rotation=45)

# Plot R2 score on a secondary y-axis
ax2 = ax1.twinx()
test_results_df.set_index('model')['r2_score'].plot(kind='line', marker='o', color='r', ax=ax2)
ax2.set_ylabel('R² Score', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax2.set_ylim(0, max(1.0, test_results_df['r2_score'].max() * 1.1))

fig.tight_layout()
plt.show()

test_results_df

In [ ]:
# --- 4. Evaluation on External Dataset (Total Time Regression) ---
import matplotlib.pyplot as plt

# --- Preprocess Test Set ---
test_results_path = os.path.join(RESULTS_DIR, "total_time_test_results2.csv")

if os.path.exists(test_results_path):
    print(f"Test results already exist at {test_results_path}, loading from file.")
    test_results_df2 = pd.read_csv(test_results_path)
    # Reload test data for detailed analysis later
    test_df = load_csv('recipes_df_test_bis.csv')
    test_df = test_df.dropna(subset=['embeddings_reg', 'times'])
    test_df['time_vec'] = test_df['times'].apply(parse_time)
    test_df['total_time'] = test_df['time_vec'].apply(lambda x: sum(x) if None not in x else None)
    valid_test_df = test_df[test_df['total_time'].notnull()]
    X_test, y_test_list = prepare_embeddings_data(valid_test_df, target_column='total_time', embedding_column='embeddings_reg')
    y_test = np.array(y_test_list).flatten()
else:
    print("Evaluating models on external recipes_df_test_bis.csv dataset for total time prediction...")
    test_df = load_csv('recipes_df_test_bis.csv')
    test_df = test_df.dropna(subset=['embeddings_reg', 'times'])
    test_df['time_vec'] = test_df['times'].apply(parse_time)
    test_df['total_time'] = test_df['time_vec'].apply(lambda x: sum(x) if None not in x else None)
    valid_test_df = test_df[test_df['total_time'].notnull()]
    X_test, y_test_list = prepare_embeddings_data(valid_test_df, target_column='total_time', embedding_column='embeddings_reg')
    y_test = np.array(y_test_list).flatten()

    def evaluate_saved_models_time(
        model_configs: list,
        X: np.ndarray,
        y: np.ndarray,
        path_start: str = 'total_time_train'
    ) -> pd.DataFrame:
        """
        Evaluate saved total time regression models on a test set.
        
        Args:
            model_configs: List of model configuration dictionaries
            X: Test features (embeddings)
            y: Test labels (total time values)
            path_start: Path prefix for loading saved models
            
        Returns:
            DataFrame with model names and their metric scores
        """
        metric_map = {
            "mean_squared_error": "mse",
            "mean_absolute_error": "mae",
            "r2_score": "r2",
        }
        
        records = []
        
        for model_cfg in model_configs:
            model_name = model_cfg['name']
            
            try:
                # Load trained model
                model = load_model(
                    model_cfg['class'],
                    model_name,
                    model_cfg['params'],
                    path_start=path_start,
                    augmentation='none'
                )
                
                print(f"Evaluating {model_name}...")
                
                # Create predictor
                predictor = SimplePredictor(
                    model=model,
                    task_type='regression',
                    batch_size=32
                )
                
                # Get predictions
                y_pred = predictor.predict(X)
                
                # Compute all metrics from the map
                scores = {
                    label: float(METRIC_REGISTRY[key](y, y_pred))
                    for label, key in metric_map.items()
                }
                
                records.append({
                    "model": model_name,
                    **scores,
                })
                print(f"  ✓ {model_name} complete")
                
            except FileNotFoundError:
                print(f"  ⚠️ Skipping {model_name}: checkpoint not found")
                continue
            except Exception as e:
                print(f"  ✗ Error evaluating {model_name}: {e}")
                import traceback
                traceback.print_exc()
                continue
        
        return pd.DataFrame.from_records(records)

    # --- Evaluate all models ---
    test_results_df2 = evaluate_saved_models_time(
        model_configs=model_configs,
        X=X_test,
        y=y_test,
        path_start='total_time_train'
    )

    # Save test results
    test_results_df2.to_csv(test_results_path, index=False)
    print("Test results saved to", test_results_path)

# --- Visualize Results ---
fig, ax1 = plt.subplots(figsize=(12, 6))
print(test_results_df2.columns)
test_results_df2.set_index('model')[['mean_squared_error', 'mean_absolute_error']].plot(kind='bar', ax=ax1)
ax1.set_ylabel('Error (MSE, MAE)')
ax1.set_xlabel('Model')
ax1.set_title("Total Time Regression Test Metrics (recipes_df_test_bis.csv)")
ax1.tick_params(axis='x', rotation=45)

# Plot R2 score on a secondary y-axis
ax2 = ax1.twinx()
test_results_df2.set_index('model')['r2_score'].plot(kind='line', marker='o', color='r', ax=ax2)
ax2.set_ylabel('R² Score', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax2.set_ylim(0, max(1.0, test_results_df2['r2_score'].max() * 1.1))

fig.tight_layout()
plt.show()

test_results_df2

In [ ]:
# --- Detailed Analysis for Best Model (Total Time Regression) ---
# Find the best model based on R2 score
best_model_name = test_results_df.loc[test_results_df['r2_score'].idxmax()]['model']
print(f"\n--- Detailed Analysis for Best Model: {best_model_name} ---")

# Get predictions for the best model again
best_model_cfg = next(cfg for cfg in model_configs if cfg['name'] == best_model_name)
model = load_model(best_model_cfg['class'], best_model_name, best_model_cfg['params'], path_start='total_time_train')
test_df = load_csv('recipes_df_test.csv')
test_df = test_df.dropna(subset=['embeddings_reg', 'times'])
test_df['time_vec'] = test_df['times'].apply(parse_time)
test_df['total_time'] = test_df['time_vec'].apply(lambda x: sum(x) if None not in x else None)
valid_test_df = test_df[test_df['total_time'].notnull()]
X_test, y_test_list = prepare_embeddings_data(valid_test_df, target_column='total_time', embedding_column='embeddings_reg')
y_test = np.array(y_test_list).flatten()

# Use SimplePredictor for inference
predictor = SimplePredictor(
    model=model,
    task_type='regression',
    device='cpu',
    batch_size=128
)

y_pred_best = predictor.predict(X_test)

# Plot regression results for total time
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual Total Time (min)")
plt.ylabel("Predicted Total Time (min)")
plt.title(f"Total Time Prediction vs. Actual for {best_model_name}")
plt.tight_layout()
plt.show()

# Print metrics for best model
mse = mean_squared_error(y_test, y_pred_best)
mae = mean_absolute_error(y_test, y_pred_best)
r2 = r2_score(y_test, y_pred_best)
print(f"Mean Squared Error: {mse:.2f}")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"R² Score: {r2:.3f}")

### Time Classification

In [ ]:
time_df = load_csv('recipes_df.csv')
def parse_time(row):
    import ast
    try:
        d = ast.literal_eval(row) if isinstance(row, str) else row
        if not isinstance(d, dict):
            return [None, None]
        prep = d.get('Preparation', '0').replace('No Time','0').replace('mins','').replace('min','').strip()
        cook = d.get('Cooking', '0').replace('No Time','0').replace('mins','').replace('min','').strip()
        try:
            prep = int(round(float(prep)))
        except Exception:
            prep = 0
        try:
            cook = int(round(float(cook)))
        except Exception:
            cook = 0
        return [prep, cook]
    except Exception:
        return [None, None]
time_df['time_vec'] = time_df['times'].apply(parse_time)
time_df['total_time'] = time_df['time_vec'].apply(lambda x: sum(x) if None not in x else None)
valid_time_df = time_df[time_df['time_vec'].apply(lambda x: None not in x) & time_df['total_time'].notnull()]

# Bin total_time: <15, 15-30, 30-60, >60
def time_bin(t):
    if t < 15:
        return 0
    elif t < 30:
        return 1
    elif t < 60:
        return 2
    else:
        return 3

valid_time_df['total_time_bin'] = valid_time_df['total_time'].apply(time_bin)

X_timec, y_timec = prepare_embeddings_data(
    valid_time_df, 
    target_column='total_time_bin', 
    embedding_column='embeddings_class'
)
# Convert y_timec to a proper 1D array
y_timec = np.array(y_timec).flatten()

X_train_timec, X_val_timec, _, y_train_timec, y_val_timec, _ = train_val_test_split(
    X_timec, 
    y_timec, 
    val_size=0.15, 
    test_size=0, 
    stratify=y_timec
)
input_dim = X_train_timec.shape[1]
output_dim = 1  # For classification, output is 1D

# Print class distribution for sanity check
import numpy as np
unique, counts = np.unique(y_train_timec, return_counts=True)
print("Train set total_time_bin class distribution:")
for cls, cnt in zip(unique, counts):
    print(f"Class {cls}: {cnt}")

In [ ]:
num_classes = len(np.unique(y_train_timec))
model_names = [
    'mlp_classifier', 'deep_mlp_classifier', 'random_forest_classifier', 'xgboost_classifier', 'lightgbm_classifier'
]
model_configs = [
    {
        'name': name,
        'class': MODEL_REGISTRY[name],
        'params': {'input_dim': input_dim, 'num_classes': num_classes} if 'mlp' in name else {}
    }
    for name in model_names
]
metrics = [METRIC_REGISTRY['f1'], METRIC_REGISTRY['recall'], METRIC_REGISTRY['precision'], METRIC_REGISTRY['accuracy']]

In [ ]:
# Train total time classification models using BenchmarkRunner
print(f"Training total time classification models on {len(X_timec)} samples...")

# Best augmentation can be included if desired
best_aug_name = 'borderline_smote'
best_augmentation = AUGMENTATION_REGISTRY[best_aug_name]

total_time_class_runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],  # Can use [best_augmentation] if desired
    metrics=metrics,
    task_type="classification",
    device="cpu",
    epochs=150,
    batch_size=32,
    use_kfold=False,
    use_class_weights=True,
    learning_rate=1e-4,
    weight_decay=5e-4,
    path_start="total_time_class_train",
    random_state=42,
)

# Run training - results are automatically saved to results/total_time_class_train/
total_time_class_runner.run(X_timec, y_timec)

print("\nTotal time classification training complete! Models and metrics saved to results/total_time_class_train/")


## 4.Evaluation

In [ ]:
test_df = load_csv('recipes_df_test.csv')
test_df = test_df.dropna(subset=['embeddings_class', 'times'])

def parse_time(row):
    import ast
    try:
        d = ast.literal_eval(row) if isinstance(row, str) else row
        if not isinstance(d, dict):
            return [None, None]
        prep = d.get('Preparation', '0').replace('No Time','0').replace('mins','').replace('min','').strip()
        cook = d.get('Cooking', '0').replace('No Time','0').replace('mins','').replace('min','').strip()
        try:
            prep = int(round(float(prep)))
        except Exception:
            prep = 0
        try:
            cook = int(round(float(cook)))
        except Exception:
            cook = 0
        return [prep, cook]
    except Exception:
        return [None, None]

test_df['time_vec'] = test_df['times'].apply(parse_time)
test_df['total_time'] = test_df['time_vec'].apply(lambda x: sum(x) if None not in x else None)

# Bin total_time: <15, 15-30, 30-60, >60
def time_bin(t):
    if t < 15:
        return 0
    elif t < 30:
        return 1
    elif t < 60:
        return 2
    else:
        return 3

valid_test_df = test_df[test_df['total_time'].notnull()]
valid_test_df['total_time_bin'] = valid_test_df['total_time'].apply(time_bin)

X_test, y_test = prepare_embeddings_data(
    valid_test_df,
    target_column='total_time_bin',
    embedding_column='embeddings_class'
)
y_test = np.array(y_test).flatten()
unique, counts = np.unique(y_test, return_counts=True)
class_dist = dict(zip(unique, counts))
print("Test set total_time_bin class distribution:")
for cls, cnt in class_dist.items():
    print(f"Class {cls}: {cnt}")

def evaluate_saved_models_time_class(
    model_configs: list,
    X: np.ndarray,
    y: np.ndarray,
    path_start: str = 'total_time_class_train'
) -> pd.DataFrame:
    """
    Evaluate saved time classification models on a test set.
    
    Args:
        model_configs: List of model configuration dictionaries
        X: Test features (embeddings)
        y: Test labels (time bins)
        path_start: Path prefix for loading saved models
        
    Returns:
        DataFrame with model names and their metric scores
    """
    metric_map = {
        "accuracy": "accuracy",
        "f1_macro": "f1",
        "precision_macro": "precision",
        "recall_macro": "recall",
    }
    
    records = []
    
    for model_cfg in model_configs:
        model_name = model_cfg['name']
        
        try:
            # Load trained model
            model = load_model(
                model_cfg['class'],
                model_name,
                model_cfg['params'],
                path_start=path_start,
                augmentation='none'
            )
            
            print(f"Evaluating {model_name}...")
            
            # Create predictor
            predictor = SimplePredictor(
                model=model,
                task_type='classification',
                batch_size=32
            )
            
            # Get probabilities for all metrics
            probs = predictor.predict_proba(X)
            
            # Compute all metrics from the map
            scores = {
                label: float(METRIC_REGISTRY[key](y, probs))
                for label, key in metric_map.items()
            }
            
            records.append({
                "model": model_name,
                **scores,
            })
            print(f"  ✓ {model_name} complete")
            
        except FileNotFoundError:
            print(f"  ⚠️ Skipping {model_name}: checkpoint not found")
            continue
        except Exception as e:
            print(f"  ✗ Error evaluating {model_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    return pd.DataFrame.from_records(records)

# Evaluate all models
print("\n" + "="*60)
print("Evaluating time classification models...")
print("="*60)
test_results_df = evaluate_saved_models_time_class(
    model_configs=model_configs,
    X=X_test,
    y=y_test,
    path_start='total_time_class_train'
)

# Save test results
os.makedirs(RESULTS_DIR, exist_ok=True)
test_results_csv = os.path.join(RESULTS_DIR, "total_time_class_test_results.csv")
test_results_df.to_csv(test_results_csv, index=False)
print(f"✓ Test results saved to {test_results_csv}")

# Plot grouped bar chart of metrics for all models
import matplotlib.pyplot as plt
test_results_df.set_index('model').plot(kind='bar', figsize=(12,6))
plt.title("Total Time Classification Test Metrics (recipes_df_test.csv)")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Metric", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Sort by accuracy and display
test_results_df.sort_values('accuracy', ascending=False)

In [ ]:
test_df = load_csv('recipes_df_test_bis.csv')
test_df = test_df.dropna(subset=['embeddings_class', 'times'])

def parse_time(row):
    import ast
    try:
        d = ast.literal_eval(row) if isinstance(row, str) else row
        if not isinstance(d, dict):
            return [None, None]
        prep = d.get('Preparation', '0').replace('No Time','0').replace('mins','').replace('min','').strip()
        cook = d.get('Cooking', '0').replace('No Time','0').replace('mins','').replace('min','').strip()
        try:
            prep = int(round(float(prep)))
        except Exception:
            prep = 0
        try:
            cook = int(round(float(cook)))
        except Exception:
            cook = 0
        return [prep, cook]
    except Exception:
        return [None, None]

test_df['time_vec'] = test_df['times'].apply(parse_time)
test_df['total_time'] = test_df['time_vec'].apply(lambda x: sum(x) if None not in x else None)

# Bin total_time: <15, 15-30, 30-60, >60
def time_bin(t):
    if t < 15:
        return 0
    elif t < 30:
        return 1
    elif t < 60:
        return 2
    else:
        return 3

valid_test_df = test_df[test_df['total_time'].notnull()]
valid_test_df['total_time_bin'] = valid_test_df['total_time'].apply(time_bin)

X_test, y_test = prepare_embeddings_data(
    valid_test_df,
    target_column='total_time_bin',
    embedding_column='embeddings_class'
)
y_test = np.array(y_test).flatten()
unique, counts = np.unique(y_test, return_counts=True)
class_dist = dict(zip(unique, counts))
print("Test set total_time_bin class distribution:")
for cls, cnt in class_dist.items():
    print(f"Class {cls}: {cnt}")

def evaluate_saved_models_time_class(
    model_configs: list,
    X: np.ndarray,
    y: np.ndarray,
    path_start: str = 'total_time_class_train'
) -> pd.DataFrame:
    """
    Evaluate saved time classification models on a test set.
    
    Args:
        model_configs: List of model configuration dictionaries
        X: Test features (embeddings)
        y: Test labels (time bins)
        path_start: Path prefix for loading saved models
        
    Returns:
        DataFrame with model names and their metric scores
    """
    metric_map = {
        "accuracy": "accuracy",
        "f1_macro": "f1",
        "precision_macro": "precision",
        "recall_macro": "recall",
    }
    
    records = []
    
    for model_cfg in model_configs:
        model_name = model_cfg['name']
        
        try:
            # Load trained model
            model = load_model(
                model_cfg['class'],
                model_name,
                model_cfg['params'],
                path_start=path_start,
                augmentation='none'
            )
            
            print(f"Evaluating {model_name}...")
            
            # Create predictor
            predictor = SimplePredictor(
                model=model,
                task_type='classification',
                batch_size=32
            )
            
            # Get probabilities for all metrics
            probs = predictor.predict_proba(X)
            
            # Compute all metrics from the map
            scores = {
                label: float(METRIC_REGISTRY[key](y, probs))
                for label, key in metric_map.items()
            }
            
            records.append({
                "model": model_name,
                **scores,
            })
            print(f"  ✓ {model_name} complete")
            
        except FileNotFoundError:
            print(f"  ⚠️ Skipping {model_name}: checkpoint not found")
            continue
        except Exception as e:
            print(f"  ✗ Error evaluating {model_name}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    return pd.DataFrame.from_records(records)

# Evaluate all models
print("\n" + "="*60)
print("Evaluating time classification models...")
print("="*60)
test_results_df = evaluate_saved_models_time_class(
    model_configs=model_configs,
    X=X_test,
    y=y_test,
    path_start='total_time_class_train'
)

# Save test results
os.makedirs(RESULTS_DIR, exist_ok=True)
test_results_csv = os.path.join(RESULTS_DIR, "total_time_class_test_results_b.csv")
test_results_df.to_csv(test_results_csv, index=False)
print(f"✓ Test results saved to {test_results_csv}")

# Plot grouped bar chart of metrics for all models
import matplotlib.pyplot as plt
test_results_df.set_index('model').plot(kind='bar', figsize=(12,6))
plt.title("Total Time Classification Test Metrics (recipes_df_test_bis.csv)")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=45, ha='right')
plt.legend(title="Metric", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Sort by accuracy and display
test_results_df.sort_values('accuracy', ascending=False)


## Conclusion
This notebook provides a full benchmarking workflow for recipe classification/regression, including augmentation selection, model comparison, and external validation. All results are reproducible and ready for further analysis.
